In [ ]:
# ============================================================
# Startup Cell: Mount Drive + Load Config
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# Add src to path
sys.path.append("/content/drive/MyDrive/DIP_Project/src")

from project_config import *

# Resolve PROJECT_ROOT from config
PROJECT_ROOT = Path(BASE_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
# ============================================================
# Cell 0: Notebook Summary
# ============================================================
#
# Purpose:
# This notebook preprocesses images for one selected dataset source.
# It reads source images from Google Drive, applies standardized
# preprocessing, saves processed images as PNG files, and writes
# an updated preprocessed metadata CSV.
#
# Design Overview:
# - This notebook processes one dataset source at a time
# - The dataset source is selected in Cell 1 using TARGET_NAME
# - Shared paths and preprocessing constants are loaded from
#   src/project_config.py
#
# Supported targets (select in Cell 3):
# - diffusiondb
# - sdxl
# - imagenet
# - coco
# - midjourney
# - openimages
#
# Inputs:
# - Source image directory on Google Drive
# - Source metadata CSV on Google Drive
#
# Processing steps:
# 1. Select target dataset using TARGET_NAME (in Cell 3)
# 2. Load source metadata CSV
# 3. Read each source image from Google Drive
# 4. Validate image
# 5. Resize image to the configured output size
# 6. Convert image to grayscale
# 7. Save image as PNG
# 8. Write updated preprocessed metadata CSV
#
# Outputs:
# - Preprocessed PNG images written to a target-specific output folder
# - Preprocessed metadata CSV written to Google Drive
#
# Notes:
# - No upload step is needed
# - No unzip step is needed
# - Images are read directly from Google Drive
# - Google Drive is mounted in Colab under:
#       /content/drive/MyDrive/
# - Output filenames follow the project naming convention:
#       [label]_[dataset]_[index].png
# - Resize dimensions, output paths, and other shared constants
#   should come from src/project_config.py
# ============================================================


In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
import glob
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


In [ ]:
# ============================================================
# Cell 3: Select Target
# ============================================================

TARGETS = [
    "diffusiondb",
    "sdxl",
    "imagenet",
    "coco",
    "midjourney",
    "openimages",
]

# Change only this line
TARGET_NAME = "openimages"

if TARGET_NAME not in TARGETS:
    raise ValueError(
        f"Invalid TARGET_NAME: {TARGET_NAME}\n"
        f"Valid options: {', '.join(TARGETS)}"
    )

print("TARGET_NAME =", TARGET_NAME)


In [ ]:
# ============================================================
# Cell 4: Target Configuration
# ============================================================

METADATA_DIR = os.path.join(PROJECT_ROOT, "data/metadata")

TARGET_CONFIGS = {
    "diffusiondb": {
        "source_image_dir": os.path.join(PROJECT_ROOT, "data/raw/DiffusionDB/images"),
        "output_dir": os.path.join(PROJECT_ROOT, "data/preprocessed/DiffusionDB"),
        "dataset_code": "diff",
        "class_label": "ai",
        "filename_label": "ai",
        "source_dataset": "DiffusionDB",
    },
    "sdxl": {
        "source_image_dir": os.path.join(PROJECT_ROOT, "data/raw/SDXL_Generated_10K/images"),
        "output_dir": os.path.join(PROJECT_ROOT, "data/preprocessed/SDXL_Generated_10K"),
        "dataset_code": "sdxl",
        "class_label": "ai",
        "filename_label": "ai",
        "source_dataset": "SDXL_Generated_10K",
    },
    "imagenet": {
        "source_image_dir": os.path.join(PROJECT_ROOT, "data/raw/ImageNet_1K_256/images"),
        "output_dir": os.path.join(PROJECT_ROOT, "data/preprocessed/ImageNet_1K_256"),
        "dataset_code": "imgn",
        "class_label": "real",
        "filename_label": "rl",
        "source_dataset": "ImageNet_1K_256",
    },
    "coco": {
        "source_image_dir": os.path.join(PROJECT_ROOT, "data/raw/MS_COCO_2017/images"),
        "output_dir": os.path.join(PROJECT_ROOT, "data/preprocessed/MS_COCO_2017"),
        "dataset_code": "coco",
        "class_label": "real",
        "filename_label": "rl",
        "source_dataset": "MS_COCO_2017",
    },
    "midjourney": {
        "source_image_dir": os.path.join(PROJECT_ROOT, "data/raw/Midjourney/images"),
        "output_dir": os.path.join(PROJECT_ROOT, "data/preprocessed/Midjourney"),
        "dataset_code": "mj",
        "class_label": "ai",
        "filename_label": "ai",
        "source_dataset": "Midjourney",
    },
    "openimages": {
        "source_image_dir": os.path.join(PROJECT_ROOT, "data/raw/OpenImages/images"),
        "output_dir": os.path.join(PROJECT_ROOT, "data/preprocessed/OpenImages"),
        "dataset_code": "open",
        "class_label": "real",
        "filename_label": "rl",
        "source_dataset": "OpenImages",
    },
}

if TARGET_NAME not in TARGET_CONFIGS:
    valid_targets = ", ".join(TARGET_CONFIGS.keys())
    raise ValueError(f"Unknown TARGET_NAME: {TARGET_NAME}. Valid options: {valid_targets}")

cfg = TARGET_CONFIGS[TARGET_NAME]

DATASET_CODE = cfg["dataset_code"]
CLASS_LABEL = cfg["class_label"]
FILENAME_LABEL = cfg["filename_label"]
SOURCE_DATASET = cfg["source_dataset"]

SOURCE_IMAGE_DIR = cfg["source_image_dir"]
SOURCE_METADATA_CSV = os.path.join(METADATA_DIR, f"{DATASET_CODE}_raw_metadata.csv")
OUTPUT_ROOT = cfg["output_dir"]
OUTPUT_IMAGE_DIR = os.path.join(OUTPUT_ROOT, "images")
OUTPUT_METADATA_CSV = os.path.join(METADATA_DIR, f"{DATASET_CODE}_preprocessed_metadata.csv")

print("SOURCE_IMAGE_DIR    =", SOURCE_IMAGE_DIR)
print("SOURCE_METADATA_CSV =", SOURCE_METADATA_CSV)
print("OUTPUT_IMAGE_DIR    =", OUTPUT_IMAGE_DIR)
print("OUTPUT_METADATA_CSV =", OUTPUT_METADATA_CSV)
print("DATASET_CODE        =", DATASET_CODE)
print("CLASS_LABEL         =", CLASS_LABEL)
print("FILENAME_LABEL      =", FILENAME_LABEL)
print("SOURCE_DATASET      =", SOURCE_DATASET)


In [ ]:
# ============================================================
# Cell 5: Validate Inputs and Create Output Folder
# ============================================================
assert os.path.exists(SOURCE_IMAGE_DIR), f"Missing source image dir: {SOURCE_IMAGE_DIR}"
assert os.path.exists(SOURCE_METADATA_CSV), f"Missing source metadata CSV: {SOURCE_METADATA_CSV}"

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)

print("PASS: source image directory exists")
print("PASS: source metadata CSV exists")
print("PASS: output image directory ready")
print("Number of source images:",
      len([f for f in os.listdir(SOURCE_IMAGE_DIR) if f.lower().endswith(('.png','.jpg','.jpeg'))]))


In [ ]:
# ============================================================
# Cell 6: Load Source Metadata
# ============================================================
df_src = pd.read_csv(SOURCE_METADATA_CSV)

print("Loaded source metadata:", SOURCE_METADATA_CSV)
print("Shape:", df_src.shape)
display(df_src.head(3))


In [ ]:
# ============================================================
# Cell 7: Image Preprocessing Utilities
# ============================================================

def preprocess_image(input_path, output_path):
    with Image.open(input_path) as img:
        img.load()

        original_format = img.format
        original_mode = img.mode
        original_width, original_height = img.size

        # Convert to grayscale
        img = img.convert("L")

        # Resize using config
        img = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)

        # Save as PNG
        img.save(output_path, format="PNG")

    return {
        "original_width": original_width,
        "original_height": original_height,
        "processed_width": TARGET_SIZE[0],
        "processed_height": TARGET_SIZE[1],
        "original_format": original_format,
        "original_mode": original_mode,
        "processed_mode": "L"
    }


In [ ]:
# ============================================================
# Cell 8: Run Preprocessing
# ============================================================
rows = []
failures = []

for idx, row in tqdm(df_src.iterrows(), total=len(df_src)):

    try:
        # ---- Input filename from source metadata ----
        source_filename = row["filename"]
        input_path = os.path.join(SOURCE_IMAGE_DIR, source_filename)

        if not os.path.exists(input_path):
            raise FileNotFoundError(f"Missing image: {input_path}")

        # ---- NEW OUTPUT FILENAME (THIS IS WHERE IT IS) ----
        output_filename = f"{FILENAME_LABEL}_{DATASET_CODE}_{idx+1:06d}.png"

        output_path = os.path.join(OUTPUT_IMAGE_DIR, output_filename)

        # ---- Preprocess image ----
        info = preprocess_image(input_path, output_path)

        # ---- Build output metadata row ----
        out_row = row.to_dict()
        out_row.update({
            "filename": output_filename,          # ← updated filename
            "original_filename": source_filename, # ← keep original reference
            "class_label": CLASS_LABEL,
            "source_dataset": SOURCE_DATASET,
            "processed_path": output_path,
            **info
        })

        rows.append(out_row)

    except Exception as e:
        failures.append({
            "row_index": idx,
            "filename": row.get("filename", ""),
            "error": str(e)
        })

print("Processed images:", len(rows))
print("Failures:", len(failures))


In [ ]:
# ============================================================
# Cell 9: Save Preprocessed Metadata CSV
# ============================================================
df_out = pd.DataFrame(rows)
df_out.to_csv(OUTPUT_METADATA_CSV, index=False)

print("Saved metadata CSV:", OUTPUT_METADATA_CSV)
print("Output shape:", df_out.shape)

if len(failures) > 0:
    df_fail = pd.DataFrame(failures)
    fail_csv = os.path.join(OUTPUT_ROOT, "preprocess_failures.csv")
    df_fail.to_csv(fail_csv, index=False)
    print("Saved failures CSV:", fail_csv)


In [ ]:
# ============================================================
# Cell 10: Validation
# ============================================================
output_images = glob.glob(os.path.join(OUTPUT_IMAGE_DIR, "*.png"))

print("Source metadata rows :", len(df_src))
print("Processed metadata rows:", len(df_out))
print("Output PNG count     :", len(output_images))

if len(df_out) == len(output_images):
    print("PASS: metadata row count matches PNG file count")
else:
    print("WARNING: metadata row count does not match PNG file count")

display(df_out.head(3))


In [ ]:
# ============================================================
# Cell 11: Visual Sanity Check
# ============================================================
sample_files = df_out["filename"].sample(n=min(4, len(df_out)), random_state=42).tolist()

plt.figure(figsize=(10, 10))

for i, fname in enumerate(sample_files, start=1):
    image_path = os.path.join(OUTPUT_IMAGE_DIR, fname)
    img = Image.open(image_path)

    plt.subplot(2, 2, i)
    plt.imshow(img, cmap="gray")
    plt.title(fname)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import pandas as pd
import re

# --- CONFIG ---
BASE_PATH = "/content/drive/MyDrive/DIP_Project/data/metadata"

DATASETS = {
    "diff": {"label": "ai"},
    "sdxl": {"label": "ai"},
    "mj": {"label": "ai"},     # if used
    "imgn": {"label": "rl"},
    "coco": {"label": "rl"},
    "open": {"label": "rl"}      # Open Images
}

STAGES = ["raw", "preprocessed"]

REQUIRED_COLUMNS = ["filename", "label", "dataset_code"]

EXPECTED_ROWS = 3000  # per dataset

# --- FUNCTION ---
def check_csv(dataset_code, stage):
    file_name = f"{dataset_code}_{stage}_metadata.csv"
    file_path = os.path.join(BASE_PATH, file_name)

    print(f"\n=== {dataset_code.upper()} [{stage.upper()}] ===")

    # 1. File existence
    if not os.path.exists(file_path):
        print(f"❌ Missing file: {file_name}")
        return None

    df = pd.read_csv(file_path)

    # 2. Row count
    row_count = len(df)
    print(f"Rows: {row_count}")
    if row_count != EXPECTED_ROWS:
        print(f"⚠️ Expected {EXPECTED_ROWS}")

    # 3. Required columns
    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        print(f"❌ Missing columns: {missing_cols}")
    else:
        print("✅ Columns OK")

    # 4. Label check
    expected_label = DATASETS[dataset_code]["label"]
    labels = df["label"].unique()
    if len(labels) == 1 and labels[0] == expected_label:
        print(f"✅ Label OK: {expected_label}")
    else:
        print(f"⚠️ Label issue: {labels}")

    # 5. Dataset code check
    codes = df["dataset_code"].unique()
    if len(codes) == 1 and codes[0] == dataset_code:
        print(f"✅ Dataset code OK")
    else:
        print(f"⚠️ Dataset code mismatch: {codes}")

    # 6. Missing filenames
    missing = df["filename"].isna().sum()
    print(f"Missing filenames: {missing}")

    # 7. Duplicate filenames
    dups = df["filename"].duplicated().sum()
    print(f"Duplicate filenames: {dups}")

    # 8. Filename format check
    pattern = rf"^{expected_label}_{dataset_code}_\d+\.png$"
    invalid = df[~df["filename"].str.match(pattern, na=False)]
    print(f"Invalid filename format: {len(invalid)}")

    # 9. Sample path check (preprocessed only)
    if stage == "preprocessed" and "processed_path" in df.columns:
        sample = df["processed_path"].dropna().head(10)
        missing_paths = [p for p in sample if not os.path.exists(p)]
        if missing_paths:
            print(f"⚠️ Missing processed files (sample): {len(missing_paths)}")
        else:
            print("✅ Processed paths OK (sample)")

    return df


# --- RUN ALL ---
raw_dfs = {}
prep_dfs = {}

for dataset in DATASETS:
    raw_dfs[dataset] = check_csv(dataset, "raw")
    prep_dfs[dataset] = check_csv(dataset, "preprocessed")


# --- CROSS-CHECK RAW vs PREPROCESSED ---
print("\n=== CROSS-CHECK RAW vs PREPROCESSED ===")

for dataset in DATASETS:
    raw_df = raw_dfs.get(dataset)
    prep_df = prep_dfs.get(dataset)

    if raw_df is None or prep_df is None:
        continue

    if len(raw_df) != len(prep_df):
        print(f"⚠️ {dataset}: row mismatch raw={len(raw_df)} preprocessed={len(prep_df)}")
    else:
        print(f"✅ {dataset}: row counts match")
